# Renewable Energy Analysis and Prediction
**PADV Project 55** — Google Colab notebook. Dataset contains observations from **1960–2015**, not 2023. Two distinct units: KTOE and renewable share (%).

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from google.colab import files
from io import BytesIO
uploaded=files.upload()  # Upload renewable_energy.csv from the project ZIP
filename=next(iter(uploaded))
df=pd.read_csv(BytesIO(uploaded[filename]))
print('Shape:',df.shape)
display(df.head());display(df.isna().sum().to_frame('Missing values'))

## Data cleaning and exploratory analysis

In [ ]:
df['Value']=pd.to_numeric(df['Value'],errors='coerce')
df['TIME']=pd.to_numeric(df['TIME'],errors='coerce')
df=df.dropna(subset=['LOCATION','MEASURE','TIME','Value'])
df=df[df.MEASURE.isin(['KTOE','PC_PRYENRGSUPPLY'])]
df=df.groupby(['LOCATION','MEASURE','TIME'],as_index=False).Value.mean()
df.TIME=df.TIME.astype(int)
print('Cleaned rows:',len(df),'Locations:',df.LOCATION.nunique(),'Years:',df.TIME.min(),'-',df.TIME.max())
display(df.groupby('MEASURE').Value.describe())

In [ ]:
from IPython.display import display
import plotly.express as px
measure='PC_PRYENRGSUPPLY'  # Change to KTOE for energy volume
subset=df[df.MEASURE==measure]
loc='AUS'
fig=px.line(subset[subset.LOCATION==loc],x='TIME',y='Value',title=f'{loc} renewable energy trend ({measure})',markers=True)
fig.show()
year=int(subset.TIME.max())
fig=px.bar(subset[subset.TIME==year].nlargest(15,'Value').sort_values('Value'),x='Value',y='LOCATION',orientation='h',title=f'Top location codes in {year} ({measure})')
fig.show()
fig=px.histogram(subset,x='Value',nbins=50,title='Distribution of renewable energy measurements')
fig.show()
fig=px.box(subset,x='TIME',y='Value',title='Yearly distribution',points=False)
fig.show()

## Machine learning — Random Forest regression
Predict each location’s renewable energy measurement using its **previous 3 consecutive years**. Chronological train/test split, so no future values are used to train on earlier years.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
base=subset.sort_values(['LOCATION','TIME']).copy()
for lag in [1,2,3]:
    previous=base[['LOCATION','TIME','Value']].copy()
    previous.TIME+=lag
    previous=previous.rename(columns={'Value':f'lag{lag}'})
    base=base.merge(previous,on=['LOCATION','TIME'],how='left')
base=base.dropna(subset=['lag1','lag2','lag3'])
features=['lag1','lag2','lag3']
years=sorted(base.TIME.unique());cut=years[max(1,int(len(years)*0.8))-1]
train=base[base.TIME<=cut];test=base[base.TIME>cut]
model=RandomForestRegressor(n_estimators=120,min_samples_leaf=3,random_state=42,n_jobs=-1)
model.fit(train[features],train.Value)
pred=model.predict(test[features])
print('Training through:',cut,'Test observations:',len(test))
print('MAE:',round(mean_absolute_error(test.Value,pred),3))
print('RMSE:',round(np.sqrt(mean_squared_error(test.Value,pred)),3))
print('R²:',round(r2_score(test.Value,pred),3))
fig=px.scatter(x=test.Value,y=pred,labels={'x':'Actual','y':'Predicted'},title='Actual vs Predicted — Holdout Years')
fig.show()

## Interactive Google Colab chart filters

In [ ]:
import ipywidgets as widgets
from IPython.display import display
location_widget=widgets.Dropdown(options=sorted(subset.LOCATION.unique()),value='AUS' if 'AUS' in subset.LOCATION.unique() else sorted(subset.LOCATION.unique())[0],description='Location:')
def show_location(location):
    s=subset[subset.LOCATION==location].sort_values('TIME')
    px.line(s,x='TIME',y='Value',markers=True,title=f'{location} historical renewable energy').show()
widgets.interact(show_location,location=location_widget);

## Conclusion
This project analyzes renewable energy measurements from 1960–2015 across location codes. Random Forest regression uses the three preceding annual observations to predict the next value. Interpret errors separately for KTOE and percentage measures. The Streamlit deployment provides a live interactive dashboard.